# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egeklc/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of a single content page for a specific client on a specific date.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

dim_clients = f"read_parquet('{REL}/dim_clients.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
fact_daily_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
fact_query_90d = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

In [2]:
con.sql(f"""
SELECT * FROM {fact_daily_march}
        LIMIT 5
""").df().columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [3]:
con.sql(f"""
SELECT * FROM {fact_daily_march}
        LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [4]:
con.sql(f"""
SELECT MIN(report_date) as start_date,
MAX(report_date) as end_date FROM {fact_daily_march}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date
0,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

|Bucket   | Fields | Reason |
|:-------:|:--------:|:----------:|
|Feature|`gsc_clicks`, `gsc_impressions`, `gsc_avg_position` when GSC data is available|Search performance and user engagement signals|
|Label |Refresh Priority Score|Proxy target used to rank pages by refresh review priority|
|Context|`report_date`,`client_hash_id`,`content_hash_id`|Identifies the client, content and observation date|
|Excluded|All GA4 columns| GA4 data is excluded because it is sparse and available for only 4.2% of observations.|

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {fact_daily_march}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [6]:
con.sql(f"""
SELECT COUNT(*) as total_rows,
COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_available_rows,
COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_available_rows,
ROUND(100.0 * COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) / COUNT(*), 1) as gsc_available_rate,
ROUND(100.0 * COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) / COUNT(*), 1) as ga4_available_rate
FROM {fact_daily_march}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,gsc_available_rate,ga4_available_rate
0,9841378,3611061,413966,36.7,4.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- The history is not balanced across all clients. GSC and GA4 data have different start dates for each client and some clients have no data for a source.
- GA4 data is avaiable for only 4.2% of observartions in the March slice. For rows where `ga4_data_available = FALSE`, zero filled GA4 columns should not be interpreted as zero engagement, but as unavailable GA4 data.


In [10]:
con.sql(f"""
SELECT gsc_data_start, ga4_data_start
FROM {dim_clients}
""").df()


,gsc_data_start,ga4_data_start
0,NaT,2026-05-22
1,NaT,NaT
2,2026-04-10,2026-04-06
3,2025-11-05,NaT
4,2025-09-24,NaT
...,...,...
99,NaT,NaT
100,2026-03-19,2026-04-24
101,NaT,NaT
102,2025-03-11,2026-03-06


## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.